In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/ai4i2020.csv")

In [2]:
X = df[
    [
        "Type",
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ]
]

y = df["Machine failure"]

X.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


In [3]:
from sklearn.model_selection import train_test_split    

In [4]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y   #when splitting the data preserve the distribution of target variable y in both training and test sets
)

## Data Preprocessing

Before training the model, we need to prepare the features for Logistic Regression.

- `Type` is a categorical feature, so we use one-hot encoding to convert it into numerical values.
- The remaining features are numerical but have different scales, so we use standardization.
- The preprocessing is fitted only on the training data to avoid data leakage.
- The same transformations are then applied to the test data.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [6]:
categorical_cols = ["Type"]

numerical_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

In [7]:
preprocessor=ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_cols),
        ("num",StandardScaler(),numerical_cols)
    ]
)

## Logistic Regression

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [9]:
lr_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [10]:
lr_cv_results = cross_validate(
    lr_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

In [11]:
for metric in scoring:
    val_scores = lr_cv_results[f"test_{metric}"]
    train_scores = lr_cv_results[f"train_{metric}"]

    print(
        f"{metric.upper():10s} "
        f"CV: {val_scores.mean():.3f} +/- {val_scores.std():.3f} | "
        f"Train: {train_scores.mean():.3f}"
    )

PRECISION  CV: 0.761 +/- 0.109 | Train: 0.747
RECALL     CV: 0.199 +/- 0.028 | Train: 0.198
F1         CV: 0.312 +/- 0.026 | Train: 0.313
ROC_AUC    CV: 0.894 +/- 0.017 | Train: 0.897
PR_AUC     CV: 0.469 +/- 0.032 | Train: 0.469


## Logistic Regression — Baseline Model

Logistic Regression is used as the first baseline model to predict machine failures. The model is trained using the selected sensor features and product type, with preprocessing performed inside a pipeline.

We evaluate the model using **5-fold stratified cross-validation** on the training set. Precision, recall, F1-score, ROC-AUC, and PR-AUC (Average Precision) are used to assess its performance because machine failures are relatively rare.

### Results and Observations

- **Precision:** 0.761 — most predicted failures are actual failures.
- **Recall:** 0.199 — the model detects only about 20% of actual failures at the default classification threshold of 0.5.
- **F1-score:** 0.312 — the balance between precision and recall is limited by the low recall.
- **ROC-AUC:** 0.894 — the model shows good ability to distinguish between failure and non-failure cases.
- **PR-AUC:** 0.469 — provides a useful measure of performance for this imbalanced classification problem.

The training and cross-validation scores are close, suggesting no substantial overfitting. However, the low recall indicates that the default threshold misses many actual failures. This baseline will be compared with other classification models before deciding whether further tuning is needed.

## Decision Tree Classifier

In [12]:
from sklearn.tree import DecisionTreeClassifier

In [13]:
dt_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

In [14]:
dt_cv_results = cross_validate(
    dt_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

In [15]:
for metric in scoring:
    val_scores = dt_cv_results[f"test_{metric}"]
    train_scores = dt_cv_results[f"train_{metric}"]

    print(
        f"{metric.upper():10s} "
        f"CV: {val_scores.mean():.3f} +/- {val_scores.std():.3f} | "
        f"Train: {train_scores.mean():.3f}"
    )

PRECISION  CV: 0.721 +/- 0.045 | Train: 1.000
RECALL     CV: 0.679 +/- 0.060 | Train: 1.000
F1         CV: 0.699 +/- 0.051 | Train: 1.000
ROC_AUC    CV: 0.835 +/- 0.030 | Train: 1.000
PR_AUC     CV: 0.503 +/- 0.071 | Train: 1.000


## Decision Tree — Baseline Model

A Decision Tree Classifier is used as the second baseline model to predict machine failures. The model uses the same selected features, preprocessing pipeline, and 5-fold stratified cross-validation strategy as Logistic Regression.

### Results and Observations

- **Precision:** 0.721 — about 72.1% of predicted failures are actual failures.
- **Recall:** 0.679 — the model detects approximately 67.9% of actual failures.
- **F1-score:** 0.699 — a better balance between precision and recall than the Logistic Regression baseline.
- **ROC-AUC:** 0.835 — the model shows good discrimination between failure and non-failure cases.
- **PR-AUC:** 0.503 — higher than the Logistic Regression baseline.

The training scores are 1.000 across all five metrics, while the cross-validation scores are lower. This indicates **significant overfitting**, suggesting that the tree has learned patterns specific to the training data that do not generalize equally well to unseen data.

Despite this overfitting, the Decision Tree achieves substantially higher recall and F1-score than Logistic Regression at the default classification threshold. We will consider controlling the tree's complexity during the hyperparameter-tuning stage.

## Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier

In [17]:
rf_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1))
])

In [18]:
rf_cv_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

In [19]:
for metric in scoring:
    val_scores = rf_cv_results[f"test_{metric}"]
    train_scores = rf_cv_results[f"train_{metric}"]

    print(
        f"{metric.upper():10s} "
        f"CV: {val_scores.mean():.3f} +/- {val_scores.std():.3f} | "
        f"Train: {train_scores.mean():.3f}"
    )

PRECISION  CV: 0.901 +/- 0.053 | Train: 1.000
RECALL     CV: 0.483 +/- 0.036 | Train: 1.000
F1         CV: 0.628 +/- 0.032 | Train: 1.000
ROC_AUC    CV: 0.960 +/- 0.011 | Train: 1.000
PR_AUC     CV: 0.771 +/- 0.026 | Train: 1.000


## Random Forest — Baseline Model

Random Forest is used as the third baseline model to predict machine failures. It combines multiple Decision Trees, each trained on a bootstrap sample of the training data. Random feature selection at each split helps introduce diversity among the trees, and their predictions are aggregated to produce the final prediction.

The model is evaluated using the same 5-fold stratified cross-validation strategy and scoring metrics as the previous models.

### Results and Observations

- **Precision:** 0.901 — approximately 90.1% of predicted failures are actual failures.
- **Recall:** 0.483 — the model detects approximately 48.3% of actual failures.
- **F1-score:** 0.628 — a better balance between precision and recall than Logistic Regression, but lower than the Decision Tree baseline.
- **ROC-AUC:** 0.960 — indicates excellent discrimination between failure and non-failure cases.
- **PR-AUC:** 0.771 — the highest PR-AUC among the three models evaluated so far.

The training scores are 1.000 across all five metrics, while the cross-validation scores are lower, indicating substantial overfitting. However, the model shows promising performance, particularly in precision, ROC-AUC, and PR-AUC.

Random Forest will be considered for hyperparameter tuning to investigate whether its generalization performance can be improved.

## XGBoost

In [20]:
from xgboost import XGBClassifier

In [21]:
xgb_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", XGBClassifier(
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    ))
])

In [22]:
xgb_results = cross_validate(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1
)

In [23]:
for metric in scoring:
    train_score = xgb_results[f"train_{metric}"]
    cv_score = xgb_results[f"test_{metric}"]

    print(
        f"{metric.upper():<12} "
        f"CV: {cv_score.mean():.3f} +/- {cv_score.std():.3f} | "
        f"Train: {train_score.mean():.3f}"
    )

PRECISION    CV: 0.803 +/- 0.071 | Train: 1.000
RECALL       CV: 0.635 +/- 0.048 | Train: 1.000
F1           CV: 0.707 +/- 0.039 | Train: 1.000
ROC_AUC      CV: 0.970 +/- 0.014 | Train: 1.000
PR_AUC       CV: 0.776 +/- 0.039 | Train: 1.000


### XGBoost Baseline — Observations and Conclusion

The XGBoost classifier was evaluated using stratified 5-fold cross-validation with the default model parameters.

**Key observations:**

- **Overfitting:** Training scores are 1.000 across all metrics, while cross-validation scores are lower, indicating a substantial generalization gap.
- **Precision (0.803):** The model's positive failure predictions are reasonably reliable.
- **Recall (0.635):** The model identifies approximately 63.5% of actual failures at the default classification threshold.
- **F1-score (0.707):** The model achieves a reasonable balance between precision and recall.
- **ROC-AUC (0.970) and PR-AUC (0.776):** Both scores indicate strong predictive discrimination, with PR-AUC being particularly useful for this imbalanced dataset.

**Conclusion:**

XGBoost demonstrates strong predictive performance but overfits in its default configuration. It will be considered for hyperparameter tuning to reduce overfitting and improve generalization. Its final suitability will be assessed by comparing its cross-validation performance with the other baseline models.

## Baseline Model Comparison

The four baseline classifiers were evaluated using stratified 5-fold cross-validation with the same training data and evaluation metrics.

| Model | Precision | Recall | F1-score | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.761 | 0.199 | 0.312 | 0.894 | 0.469 |
| Decision Tree | 0.721 | 0.679 | 0.699 | 0.835 | 0.503 |
| Random Forest | 0.901 | 0.483 | 0.628 | 0.960 | 0.771 |
| XGBoost | 0.803 | 0.635 | 0.707 | 0.970 | 0.776 |

*Note: Values represent mean cross-validation scores. Standard deviations are omitted here for readability.*

### Key Observations

- **Logistic Regression:** Achieves relatively high precision but has low recall, missing many actual machine failures at the default classification threshold.
- **Decision Tree:** Achieves the highest recall among the four baselines but has lower ROC-AUC and PR-AUC than the ensemble models.
- **Random Forest:** Achieves the highest precision and strong ROC-AUC and PR-AUC, but its recall is lower than those of the Decision Tree and XGBoost.
- **XGBoost:** Achieves the highest mean F1-score, ROC-AUC, and PR-AUC among the four models, making it a promising candidate for further optimization.

### Overall Conclusion

XGBoost and Random Forest are promising candidates based on their ranking performance, while the Decision Tree provides stronger recall at the default threshold. All three tree-based models show signs of overfitting based on the gap between training and cross-validation scores.

The next step is to tune Random Forest and XGBoost to reduce overfitting and improve generalization. Model selection should consider the trade-off between precision and recall, as missing actual machine failures may be costly in predictive maintenance.

In [24]:
from sklearn.model_selection import RandomizedSearchCV

xgb_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", XGBClassifier(
        random_state=42,
        n_jobs=1,
        eval_metric="logloss"
    ))
])

param_distributions = {
    "classifier__max_depth": [2, 3, 4, 5, 6],
    "classifier__min_child_weight": [1, 3, 5, 7],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "classifier__reg_lambda": [1, 5, 10],
    "classifier__gamma": [0, 0.1, 0.3, 0.5]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

xgb_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__colsample_bytree': [0.7, 0.8, ...], 'classifier__gamma': [0, 0.1, ...], 'classifier__learning_rate': [0.01, 0.03, ...], 'classifier__max_depth': [2, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be co

In [25]:
print("Best CV PR-AUC:", round(xgb_search.best_score_, 3))

print("\nBest parameters:")
for param, value in xgb_search.best_params_.items():
    print(f"{param}: {value}")

Best CV PR-AUC: 0.799

Best parameters:
classifier__subsample: 0.9
classifier__reg_lambda: 1
classifier__n_estimators: 100
classifier__min_child_weight: 1
classifier__max_depth: 6
classifier__learning_rate: 0.05
classifier__gamma: 0.3
classifier__colsample_bytree: 0.9


In [26]:
best_index = xgb_search.best_index_
results = xgb_search.cv_results_

print(
    "Mean training PR-AUC:",
    round(results["mean_train_score"][best_index], 3)
)

print(
    "Mean CV PR-AUC:",
    round(results["mean_test_score"][best_index], 3)
)

Mean training PR-AUC: 0.974
Mean CV PR-AUC: 0.799


In [27]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions_2 = {
    "classifier__max_depth": [2, 3, 4, 5],
    "classifier__min_child_weight": [3, 5, 7, 10],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__subsample": [0.7, 0.8, 0.9],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9],
    "classifier__reg_lambda": [1, 5, 10],
    "classifier__gamma": [0.1, 0.3, 0.5, 1.0]
}

xgb_search_2 = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions_2,
    n_iter=30,
    scoring="average_precision",
    cv=cv,
    random_state=43,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

xgb_search_2.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__colsample_bytree': [0.7, 0.8, ...], 'classifier__gamma': [0.1, 0.3, ...], 'classifier__learning_rate': [0.01, 0.03, ...], 'classifier__max_depth': [2, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",43
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be 

In [28]:
print("First search CV PR-AUC:", round(xgb_search.best_score_, 3))
print("Second search CV PR-AUC:", round(xgb_search_2.best_score_, 3))

best_idx_2 = xgb_search_2.best_index_
results_2 = xgb_search_2.cv_results_

print(
    "Second search training PR-AUC:",
    round(results_2["mean_train_score"][best_idx_2], 3)
)

print(
    "Second search validation PR-AUC:",
    round(results_2["mean_test_score"][best_idx_2], 3)
)

print("\nBest parameters from second search:")
print(xgb_search_2.best_params_)

First search CV PR-AUC: 0.799
Second search CV PR-AUC: 0.776
Second search training PR-AUC: 0.935
Second search validation PR-AUC: 0.776

Best parameters from second search:
{'classifier__subsample': 0.9, 'classifier__reg_lambda': 1, 'classifier__n_estimators': 300, 'classifier__min_child_weight': 5, 'classifier__max_depth': 4, 'classifier__learning_rate': 0.05, 'classifier__gamma': 0.3, 'classifier__colsample_bytree': 0.8}


### Second XGBoost Hyperparameter Search

A second randomized search was conducted using a more focused parameter space, emphasizing shallower trees and stronger constraints on tree splitting. The objective remained maximizing mean cross-validation average precision (PR-AUC).

**Observations:**

- The best CV PR-AUC decreased from 0.799 in the first search to 0.776 in the second search.
- The training PR-AUC decreased from 0.974 to 0.935, indicating that the selected model fits the training data less closely.
- The training–validation PR-AUC gap decreased from 0.175 to 0.159, suggesting a modest reduction in the generalization gap.

**Conclusion:**

The second search did not improve cross-validation PR-AUC, although it reduced the training–validation gap slightly. This suggests a trade-off between model complexity and validation performance. The first search currently has the higher CV PR-AUC, but neither configuration will be declared the final winner until their precision, recall, F1-score and ROC-AUC are compared under the same evaluation procedure.

In [29]:
from sklearn.model_selection import cross_validate

xgb_candidates = {
    "First Search": xgb_search.best_estimator_,
    "Second Search": xgb_search_2.best_estimator_
}

xgb_comparison = {}

for name, model in xgb_candidates.items():
    results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1
    )

    xgb_comparison[name] = results

    print(f"\n{name}")
    print("-" * 55)

    for metric in scoring:
        cv_scores = results[f"test_{metric}"]
        train_scores = results[f"train_{metric}"]

        print(
            f"{metric.upper():<12} "
            f"CV: {cv_scores.mean():.3f} "
            f"+/- {cv_scores.std():.3f} | "
            f"Train: {train_scores.mean():.3f}"
        )


First Search
-------------------------------------------------------
PRECISION    CV: 0.875 +/- 0.055 | Train: 0.989
RECALL       CV: 0.639 +/- 0.048 | Train: 0.852
F1           CV: 0.736 +/- 0.025 | Train: 0.916
ROC_AUC      CV: 0.976 +/- 0.012 | Train: 0.997
PR_AUC       CV: 0.799 +/- 0.041 | Train: 0.974

Second Search
-------------------------------------------------------
PRECISION    CV: 0.834 +/- 0.052 | Train: 0.951
RECALL       CV: 0.583 +/- 0.052 | Train: 0.785
F1           CV: 0.685 +/- 0.043 | Train: 0.860
ROC_AUC      CV: 0.975 +/- 0.013 | Train: 0.994
PR_AUC       CV: 0.776 +/- 0.027 | Train: 0.935


In [31]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import cross_val_predict

best_xgb_pipeline = xgb_search.best_estimator_

oof_probabilities = cross_val_predict(
    best_xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

print("Number of OOF predictions:", len(oof_probabilities))
print("Number of training examples:", len(y_train))

Number of OOF predictions: 8000
Number of training examples: 8000


In [32]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

thresholds = np.arange(0.10, 0.61, 0.05)

threshold_results = []

for threshold in thresholds:
    y_pred = (oof_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_train, y_pred, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(
            y_train, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_train, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_train, y_pred, zero_division=0
        ),
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "true_negatives": tn
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(3))

,threshold,precision,recall,f1,false_positives,false_negatives,true_positives,true_negatives
0,0.10,0.500,0.845,0.628,229,42,229,7500
1,0.15,0.599,0.827,0.695,150,47,224,7579
2,0.20,0.667,0.812,0.732,110,51,220,7619
3,0.25,0.725,0.779,0.751,80,60,211,7649
4,0.30,0.761,0.753,0.757,64,67,204,7665
5,0.35,0.778,0.712,0.744,55,78,193,7674
6,0.40,0.805,0.686,0.741,45,85,186,7684
7,0.45,0.838,0.668,0.743,35,90,181,7694
8,0.50,0.869,0.638,0.736,26,98,173,7703
9,0.55,0.880,0.598,0.712,22,109,162,7707


In [33]:
print("Threshold comparison, sorted by recall:\n")

display(
    threshold_df.sort_values(
        by=["recall", "precision"],
        ascending=[False, False]
    ).round(3)
)

Threshold comparison, sorted by recall:



,threshold,precision,recall,f1,false_positives,false_negatives,true_positives,true_negatives
0,0.10,0.500,0.845,0.628,229,42,229,7500
1,0.15,0.599,0.827,0.695,150,47,224,7579
2,0.20,0.667,0.812,0.732,110,51,220,7619
3,0.25,0.725,0.779,0.751,80,60,211,7649
4,0.30,0.761,0.753,0.757,64,67,204,7665
5,0.35,0.778,0.712,0.744,55,78,193,7674
6,0.40,0.805,0.686,0.741,45,85,186,7684
7,0.45,0.838,0.668,0.743,35,90,181,7694
8,0.50,0.869,0.638,0.736,26,98,173,7703
9,0.55,0.880,0.598,0.712,22,109,162,7707


### Classification Threshold Analysis

Out-of-fold predicted probabilities from the first XGBoost search's best estimator were used to evaluate classification thresholds from 0.10 to 0.60. This analysis used the training dataset only; the held-out test set was not used.

**Key observations:**

- Lowering the threshold generally increases recall and reduces the number of missed machine failures.
- At a threshold of 0.10, recall reaches 0.845, with 42 false negatives and 229 false positives.
- At the default threshold of 0.50, recall is 0.638, with 98 false negatives and 26 false positives.
- The results illustrate the trade-off between detecting more failures and generating additional false alarms.

**Conclusion:**

Because missed machine failures may be more costly than unnecessary inspections, higher recall is an important objective for this project. A threshold of 0.10 is a candidate for further consideration, but the final threshold will be selected after considering the operational cost trade-off and comparing the tuned models. The held-out test set will remain untouched until final evaluation.

## Random Forest Hyperparameter Tuning

The baseline Random Forest achieved strong precision and ranking performance, with a cross-validation precision of 0.901, ROC-AUC of 0.960, and PR-AUC of 0.771. However, its recall was 0.483 at the default classification threshold, indicating that many actual machine failures were missed.

Hyperparameter tuning will investigate whether controlling tree complexity and leaf size can improve generalization and failure detection.

The search will use stratified 5-fold cross-validation on the training data, with average precision (PR-AUC) as the primary optimization metric. Precision, recall, F1-score, and ROC-AUC will also be evaluated.

The held-out test set will remain untouched until final model evaluation.

In [34]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ))
])

rf_param_distributions = {
    "classifier__n_estimators": [200, 300, 500],
    "classifier__max_depth": [None, 5, 10, 15, 20],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 4, 8],
    "classifier__max_features": ["sqrt", 0.7, 1.0],
    "classifier__class_weight": [None, "balanced", "balanced_subsample"]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_distributions,
    n_iter=30,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

rf_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__class_weight': [None, 'balanced', ...], 'classifier__max_depth': [None, 5, ...], 'classifier__max_features': ['sqrt', 0.7, ...], 'classifier__min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training

In [35]:
print("Best CV PR-AUC:", round(rf_search.best_score_, 3))
print("\nBest parameters:")

for param, value in rf_search.best_params_.items():
    print(f"{param}: {value}")

best_idx = rf_search.best_index_
rf_results = rf_search.cv_results_

print(
    "\nMean training PR-AUC:",
    round(rf_results["mean_train_score"][best_idx], 3)
)

print(
    "Mean CV PR-AUC:",
    round(rf_results["mean_test_score"][best_idx], 3)
)

Best CV PR-AUC: 0.821

Best parameters:
classifier__n_estimators: 500
classifier__min_samples_split: 2
classifier__min_samples_leaf: 1
classifier__max_features: 1.0
classifier__max_depth: 20
classifier__class_weight: None

Mean training PR-AUC: 1.0
Mean CV PR-AUC: 0.821


### Random Forest Hyperparameter Tuning — Observations

Randomized hyperparameter search was performed using stratified 5-fold cross-validation on the training dataset, with average precision (PR-AUC) as the optimization metric.

**Best configuration:**

- `n_estimators`: 500
- `max_depth`: 20
- `min_samples_split`: 2
- `min_samples_leaf`: 1
- `max_features`: 1.0
- `class_weight`: None

**Key observations:**

- The best mean CV PR-AUC was 0.821.
- The mean training PR-AUC was 1.000, indicating a substantial training–validation gap.
- The tuned model achieved a higher CV PR-AUC than the first tuned XGBoost configuration (0.799), making it a promising candidate for further evaluation.

**Conclusion:**

The tuned Random Forest shows strong ranking performance but still exhibits signs of overfitting. Its precision, recall, F1-score and ROC-AUC must be evaluated before comparing it fairly with XGBoost. The held-out test set remains untouched.

In [36]:
rf_best_pipeline = rf_search.best_estimator_

rf_tuned_results = cross_validate(
    rf_best_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1
)

for metric in scoring:
    cv_scores = rf_tuned_results[f"test_{metric}"]
    train_scores = rf_tuned_results[f"train_{metric}"]

    print(
        f"{metric.upper():<12} "
        f"CV: {cv_scores.mean():.3f} "
        f"+/- {cv_scores.std():.3f} | "
        f"Train: {train_scores.mean():.3f}"
    )

PRECISION    CV: 0.875 +/- 0.035 | Train: 1.000
RECALL       CV: 0.686 +/- 0.065 | Train: 1.000
F1           CV: 0.767 +/- 0.044 | Train: 1.000
ROC_AUC      CV: 0.972 +/- 0.011 | Train: 1.000
PR_AUC       CV: 0.821 +/- 0.028 | Train: 1.000


In [37]:
from sklearn.model_selection import cross_val_predict

rf_best_pipeline = rf_search.best_estimator_

rf_oof_probabilities = cross_val_predict(
    rf_best_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

print("OOF predictions:", len(rf_oof_probabilities))

OOF predictions: 8000


In [38]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

rf_thresholds = np.arange(0.05, 0.61, 0.05)
rf_threshold_results = []

for threshold in rf_thresholds:
    rf_pred = (
        rf_oof_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_train,
        rf_pred,
        labels=[0, 1]
    ).ravel()

    rf_threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(
            y_train, rf_pred, zero_division=0
        ),
        "recall": recall_score(
            y_train, rf_pred, zero_division=0
        ),
        "f1": f1_score(
            y_train, rf_pred, zero_division=0
        ),
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "true_negatives": tn
    })

rf_threshold_df = pd.DataFrame(rf_threshold_results)

display(rf_threshold_df.round(3))

,threshold,precision,recall,f1,false_positives,false_negatives,true_positives,true_negatives
0,0.05,0.355,0.904,0.510,445,26,245,7284
1,0.10,0.443,0.856,0.584,292,39,232,7437
2,0.15,0.525,0.841,0.647,206,43,228,7523
3,0.20,0.602,0.815,0.693,146,50,221,7583
4,0.25,0.650,0.782,0.710,114,59,212,7615
5,0.30,0.691,0.760,0.724,92,65,206,7637
6,0.35,0.757,0.745,0.751,65,69,202,7664
7,0.40,0.805,0.716,0.758,47,77,194,7682
8,0.45,0.867,0.697,0.773,29,82,189,7700
9,0.50,0.870,0.690,0.770,28,84,187,7701


In [39]:
display(
    rf_threshold_df.sort_values(
        by=["recall", "precision"],
        ascending=[False, False]
    ).round(3)
)

,threshold,precision,recall,f1,false_positives,false_negatives,true_positives,true_negatives
0,0.05,0.355,0.904,0.510,445,26,245,7284
1,0.10,0.443,0.856,0.584,292,39,232,7437
2,0.15,0.525,0.841,0.647,206,43,228,7523
3,0.20,0.602,0.815,0.693,146,50,221,7583
4,0.25,0.650,0.782,0.710,114,59,212,7615
5,0.30,0.691,0.760,0.724,92,65,206,7637
6,0.35,0.757,0.745,0.751,65,69,202,7664
7,0.40,0.805,0.716,0.758,47,77,194,7682
8,0.45,0.867,0.697,0.773,29,82,189,7700
9,0.50,0.870,0.690,0.770,28,84,187,7701


### Random Forest Classification Threshold Analysis

Out-of-fold predicted probabilities from the tuned Random Forest model were used to evaluate classification thresholds from 0.05 to 0.60. Only the training dataset was used.

**Key observations:**

- At a threshold of 0.05, recall reached 0.904, with 26 false negatives and 445 false positives.
- At the default threshold of 0.50, recall was 0.690, with 84 false negatives and 28 false positives.
- Lower thresholds increased failure detection but generated more false alarms.
- Compared with XGBoost at thresholds of 0.10 and 0.20, Random Forest achieved slightly higher recall but also generated more false positives.

**Conclusion:**

Random Forest can achieve high recall with a lower classification threshold, but this comes at the cost of additional false alarms. Model selection will consider both missed failures and false alarms rather than maximizing recall alone. The final model and threshold will be selected after comparing the cross-validation results and operational trade-offs. The held-out test set remains untouched.

In [40]:
import pandas as pd

# CV metrics already calculated for the baseline models
baseline_results = {
    "Logistic Regression": {
        "Precision": 0.761, "Recall": 0.199,
        "F1": 0.312, "ROC-AUC": 0.894, "PR-AUC": 0.469
    },
    "Decision Tree": {
        "Precision": 0.721, "Recall": 0.679,
        "F1": 0.699, "ROC-AUC": 0.835, "PR-AUC": 0.503
    },
    "Random Forest (Baseline)": {
        "Precision": 0.901, "Recall": 0.483,
        "F1": 0.628, "ROC-AUC": 0.960, "PR-AUC": 0.771
    },
    "XGBoost (Baseline)": {
        "Precision": 0.803, "Recall": 0.635,
        "F1": 0.707, "ROC-AUC": 0.970, "PR-AUC": 0.776
    }
}

# Use the actual CV results calculated for the tuned models
tuned_results = {}

for name, results in [
    ("XGBoost (Tuned)", xgb_comparison["First Search"]),
    ("Random Forest (Tuned)", rf_tuned_results)
]:
    tuned_results[name] = {
        "Precision": results["test_precision"].mean(),
        "Recall": results["test_recall"].mean(),
        "F1": results["test_f1"].mean(),
        "ROC-AUC": results["test_roc_auc"].mean(),
        "PR-AUC": results["test_pr_auc"].mean()
    }

all_results = {**baseline_results, **tuned_results}

comparison_df = pd.DataFrame(all_results).T
display(comparison_df.round(3))

,Precision,Recall,F1,ROC-AUC,PR-AUC
Logistic Regression,0.761,0.199,0.312,0.894,0.469
Decision Tree,0.721,0.679,0.699,0.835,0.503
Random Forest (Baseline),0.901,0.483,0.628,0.960,0.771
XGBoost (Baseline),0.803,0.635,0.707,0.970,0.776
XGBoost (Tuned),0.875,0.639,0.736,0.976,0.799
Random Forest (Tuned),0.875,0.686,0.767,0.972,0.821


In [41]:
xgb_threshold_comparison = threshold_df[
    ["threshold", "precision", "recall",
     "false_positives", "false_negatives"]
].copy()

xgb_threshold_comparison.insert(
    0, "model", "XGBoost"
)

rf_threshold_comparison = rf_threshold_df[
    ["threshold", "precision", "recall",
     "false_positives", "false_negatives"]
].copy()

rf_threshold_comparison.insert(
    0, "model", "Random Forest"
)

threshold_comparison_df = pd.concat(
    [xgb_threshold_comparison, rf_threshold_comparison],
    ignore_index=True
)

display(threshold_comparison_df.round(3))

,model,threshold,precision,recall,false_positives,false_negatives
0,XGBoost,0.10,0.500,0.845,229,42
1,XGBoost,0.15,0.599,0.827,150,47
2,XGBoost,0.20,0.667,0.812,110,51
3,XGBoost,0.25,0.725,0.779,80,60
4,XGBoost,0.30,0.761,0.753,64,67
5,XGBoost,0.35,0.778,0.712,55,78
6,XGBoost,0.40,0.805,0.686,45,85
7,XGBoost,0.45,0.838,0.668,35,90
8,XGBoost,0.50,0.869,0.638,26,98
9,XGBoost,0.55,0.880,0.598,22,109


In [42]:
selected_thresholds = threshold_comparison_df[
    threshold_comparison_df["threshold"].isin([0.10, 0.20, 0.50])
].sort_values(["threshold", "model"])

display(selected_thresholds.round(3))

,model,threshold,precision,recall,false_positives,false_negatives
12,Random Forest,0.1,0.443,0.856,292,39
0,XGBoost,0.1,0.500,0.845,229,42
14,Random Forest,0.2,0.602,0.815,146,50
2,XGBoost,0.2,0.667,0.812,110,51
20,Random Forest,0.5,0.870,0.690,28,84
8,XGBoost,0.5,0.869,0.638,26,98


In [43]:
recall_target = 0.80

xgb_candidates = threshold_df[
    threshold_df["recall"] >= recall_target
].sort_values("false_positives")

rf_candidates = rf_threshold_df[
    rf_threshold_df["recall"] >= recall_target
].sort_values("false_positives")

print("XGBoost: thresholds meeting recall target")
display(xgb_candidates.round(3))

print("Random Forest: thresholds meeting recall target")
display(rf_candidates.round(3))

XGBoost: thresholds meeting recall target


,threshold,precision,recall,f1,false_positives,false_negatives,true_positives,true_negatives
2,0.20,0.667,0.812,0.732,110,51,220,7619
1,0.15,0.599,0.827,0.695,150,47,224,7579
0,0.10,0.500,0.845,0.628,229,42,229,7500


Random Forest: thresholds meeting recall target


,threshold,precision,recall,f1,false_positives,false_negatives,true_positives,true_negatives
3,0.20,0.602,0.815,0.693,146,50,221,7583
2,0.15,0.525,0.841,0.647,206,43,228,7523
1,0.10,0.443,0.856,0.584,292,39,232,7437
0,0.05,0.355,0.904,0.510,445,26,245,7284


### Provisional Model and Threshold Selection

A provisional recall target of 0.80 was used to compare the threshold results for the tuned XGBoost and Random Forest models.

Among the tested thresholds meeting this target, XGBoost at a threshold of 0.20 achieved a recall of 0.812, precision of 0.667, 110 false positives and 51 false negatives. Random Forest at the same threshold achieved a recall of 0.815, precision of 0.602, 146 false positives and 50 false negatives.

XGBoost generated 36 fewer false positives while missing one additional failure. Given the importance of limiting unnecessary alarms while maintaining high failure detection, XGBoost at a threshold of 0.20 is selected as the provisional candidate.

This decision is based on out-of-fold predictions from the training data. The threshold and model will be evaluated on the untouched test set during final evaluation.

In [44]:
from sklearn.base import clone

final_model = clone(xgb_search.best_estimator_)
final_model.fit(X_train, y_train)

print("Final XGBoost model trained on all training examples.")
print("Training examples:", len(X_train))
print("Test examples reserved for final evaluation:", len(X_test))

Final XGBoost model trained on all training examples.
Training examples: 8000
Test examples reserved for final evaluation: 2000


In [45]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Predict failure probabilities on the held-out test set
test_probabilities = final_model.predict_proba(X_test)[:, 1]

# Apply our selected provisional threshold
final_threshold = 0.20
y_test_pred = (test_probabilities >= final_threshold).astype(int)

# Calculate confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_test, y_test_pred, labels=[0, 1]
).ravel()

# Calculate evaluation metrics
test_metrics = {
    "Precision": precision_score(y_test, y_test_pred, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, test_probabilities),
    "PR-AUC": average_precision_score(y_test, test_probabilities)
}

print("Final Test Metrics")
print("-" * 35)

for metric, value in test_metrics.items():
    print(f"{metric:<12}: {value:.3f}")

print("\nConfusion Matrix Counts")
print("True negatives :", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives :", tp)

print("\nClassification Report")
print(classification_report(
    y_test, y_test_pred,
    target_names=["No Failure", "Failure"],
    zero_division=0
))

Final Test Metrics
-----------------------------------
Precision   : 0.639
Recall      : 0.779
F1-score    : 0.702
ROC-AUC     : 0.979
PR-AUC      : 0.843

Confusion Matrix Counts
True negatives : 1902
False positives: 30
False negatives: 15
True positives : 53

Classification Report
              precision    recall  f1-score   support

  No Failure       0.99      0.98      0.99      1932
     Failure       0.64      0.78      0.70        68

    accuracy                           0.98      2000
   macro avg       0.82      0.88      0.85      2000
weighted avg       0.98      0.98      0.98      2000



## Model Evaluation Summary and Explainability

### Summary of Model Development

Four baseline classification models—Logistic Regression, Decision Tree, Random Forest, and XGBoost—were evaluated using stratified 5-fold cross-validation. Hyperparameter tuning was then performed for XGBoost and Random Forest using randomized search, with average precision (PR-AUC) as the primary optimization metric.

Classification threshold analysis was conducted using out-of-fold predictions to study the trade-off between detecting machine failures and generating false alarms. Based on the cross-validation results and our preference for higher failure detection with manageable false alarms, XGBoost was provisionally selected with a classification threshold of 0.20.

### Final Test Evaluation

The selected XGBoost model was fitted on the complete training set and evaluated on the held-out test set.

| Metric | Test Score |
|---|---:|
| Precision | 0.639 |
| Recall | 0.779 |
| F1-score | 0.702 |
| ROC-AUC | 0.979 |
| PR-AUC | 0.843 |

The model detected 53 of the 68 actual failures and missed 15, while generating 30 false alarms. The test recall was slightly below our provisional target of 0.80. These results highlight the trade-off between failure detection and unnecessary alerts.

### Motivation for Model Explainability

Although the model demonstrates promising ranking performance, its predictions alone do not explain why a machine is classified as likely to fail. Understanding which input features influence its predictions can help us interpret its behaviour and identify potentially useful signals for maintenance planning.

The next stage uses **SHAP (SHapley Additive exPlanations)** to investigate feature contributions to the trained model's predictions. This analysis will help us understand the model's behaviour, while distinguishing predictive associations from causal explanations.